In [ ]:
import pandas as pd
import re
import html

# 1. Inspect the dataset
print("--- STEP 1: INITIAL INSPECTION ---")
df = pd.read_csv("processed_data (1).csv", on_bad_lines='skip')
print(f"Dataset Shape: {df.shape}")
print(f"Column Names: {list(df.columns)}")
print("\nFirst 10 rows:")
print(df.head(10))

# 2. Check data quality
print("\n--- STEP 2: QUALITY AUDIT ---")
print(f"Missing Values:\n{df.isnull().sum()}")
print(f"Duplicate Rows: {df.duplicated().sum()}")
print(f"Label Distribution:\n{df['label'].value_counts()}")

# 3. Clean the dataset
def advanced_clean(text):
    if pd.isna(text) or not isinstance(text, str) or text.strip() == "":
        return None

    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = html.unescape(text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s.,!?$@]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    if text == "":
        return None
    return text

print("\n--- STEP 3: CLEANING DATASET ---")
# Using 'clean_text' as the source column based on your previous data structure
df['text'] = df['clean_text'].apply(advanced_clean)

# Remove rows with missing email text
df = df.dropna(subset=['text'])

# Remove duplicate emails
df = df.drop_duplicates(subset=['text'])

# 4. Verify labels
print("\n--- STEP 4: LABEL VERIFICATION ---")
# Confirm labels are binary (0 = safe, 1 = phishing)
df = df[df['label'].isin([0, 1, 0.0, 1.0])].copy()
df['label'] = df['label'].astype(int)
print(f"Rows for Safe (0): {len(df[df['label'] == 0])}")
print(f"Rows for Phishing (1): {len(df[df['label'] == 1])}")

# 5. Generate a final cleaned dataset
final_df = df[['text', 'label']]
final_df.to_csv('senseflow_clean_dataset.csv', index=False)
print("\n✅ SUCCESS: senseflow_clean_dataset.csv generated successfully.")

# Final dataset analysis
print("\n--- STEP 5: FINAL DATASET ANALYSIS ---")
print("Average text length:")
print(final_df['text'].apply(len).describe())

--- STEP 1: INITIAL INSPECTION ---
Dataset Shape: (16568, 2)
Column Names: ['clean_text', 'label']

First 10 rows:
                                          clean_text  label
0  frommr james ngola\nconfidential tel 233275879...      1
1  dear friend\n\ni am mr ben suleman a custom of...      1
2  from his royal majesty hrm crown ruler of elem...      1
3  from his royal majesty hrm crown ruler of elem...      1
4  dear sir \n \nit is with a heart full of hope ...      1
5  attention                                    \...      1
6  dear sir\n\ni am barrister tunde dosumu san so...      1
7  from william drallo\nconfidential tel 23324204...      1
8  challenge securities ltd\nlagos nigeria\n\n\n\...      1
9  dear sir\n\ni am barrister tunde dosumu san so...      1

--- STEP 2: QUALITY AUDIT ---
Missing Values:
clean_text    11
label          0
dtype: int64
Duplicate Rows: 273
Label Distribution:
label
1    11925
0     4643
Name: count, dtype: int64

--- STEP 3: CLEANING DATASET ---

--

In [ ]:
import pandas as pd

# Load the clean data
df = pd.read_csv('senseflow_clean_dataset.csv')

# The AI can only read about 512 words at a time anyway.
# This chops every email down to the first 512 words.
df['text'] = df['text'].apply(lambda x: ' '.join(str(x).split()[:512]))

# Save the final, ML-ready file
df.to_csv('senseflow_ready_for_ai.csv', index=False)

print("✅ Monster emails destroyed. Data is officially ready for training.")

✅ Monster emails destroyed. Data is officially ready for training.


In [ ]:
import pandas as pd

# 1. Load the dataset we just cleaned
df = pd.read_csv('senseflow_clean_dataset.csv')

# 2. Filter out the monsters (Keep only emails under 50,000 characters)
df = df[df['text'].apply(lambda x: len(str(x)) < 50000)]

# 3. Overwrite the file with the truly clean data
df.to_csv('senseflow_clean_dataset.csv', index=False)

print("✅ Monster emails eradicated. Let's look at the new stats:")
print(df['text'].apply(lambda x: len(str(x))).describe())

✅ Monster emails eradicated. Let's look at the new stats:
count    16135.000000
mean      1274.142919
std       2091.340124
min          1.000000
25%        277.000000
50%        612.000000
75%       1674.500000
max      46332.000000
Name: text, dtype: float64


In [ ]:
import pandas as pd

print("--- POST-PHASE 2b VERIFICATION REPORT ---")
# 1. Load the final, clean dataset
df = pd.read_csv('senseflow_clean_dataset.csv')

# 2. Print the final row count to ensure we didn't delete too much
print(f"Total safe rows remaining: {len(df)}")

# 3. Print the length stats to PROVE the 4-million-character monster is dead
print("\nFinal Text Length Statistics:")
print(df['text'].apply(lambda x: len(str(x))).describe())

--- POST-PHASE 2b VERIFICATION REPORT ---
Total safe rows remaining: 5609

Final Text Length Statistics:
count     5609.000000
mean      1866.451239
std       1744.789260
min         21.000000
25%        711.000000
50%       1691.000000
75%       2690.000000
max      38340.000000
Name: text, dtype: float64


In [ ]:
import pandas as pd
import re
import html

print("--- SENSEFLOW LEAK DETECTOR ---")

# 1. Check the starting file
file_name = "processed_data (1).csv" # Change to "processed_data.csv" if needed
df = pd.read_csv(file_name, on_bad_lines='skip', engine='python')
print(f"1. Starting Raw Data: {len(df)} rows")

# 2. Check blanks
def advanced_clean(text):
    if pd.isna(text) or not isinstance(text, str) or text.strip() == "": return None
    text = str(text).lower()
    text = re.sub(r'<.*?>', '', text)
    text = html.unescape(text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s.,!?$@]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

input_col = 'clean_text' if 'clean_text' in df.columns else df.columns[0]
df['text'] = df[input_col].apply(advanced_clean)

df_no_blanks = df.dropna(subset=['text'])
print(f"2. After dropping blanks: {len(df_no_blanks)} rows")

# 3. Check Duplicates (The prime suspect)
df_no_dupes = df_no_blanks.drop_duplicates(subset=['text'])
print(f"3. After dropping duplicates: {len(df_no_dupes)} rows")

# 4. Check the Monster Filter
df_final = df_no_dupes[df_no_dupes['text'].apply(len) < 50000]
print(f"4. After killing the >50k monster: {len(df_final)} rows")

--- SENSEFLOW LEAK DETECTOR ---
1. Starting Raw Data: 16565 rows
2. After dropping blanks: 16554 rows
3. After dropping duplicates: 16149 rows
4. After killing the >50k monster: 16135 rows


In [ ]:
# Save this perfect, 16,135-row dataset permanently
df_final.to_csv('senseflow_clean_dataset.csv', index=False)
print("✅ PERFECT DATASET SAVED! The SenseFlow Data Phase is officially 100% complete.")

✅ PERFECT DATASET SAVED! The SenseFlow Data Phase is officially 100% complete.


In [ ]:
import pandas as pd

print("=== SENSEFLOW: FINAL DATASET ANALYTICS REPORT ===")

# 1. Load the officially locked-in dataset
try:
    df = pd.read_csv('senseflow_clean_dataset.csv')
    print("✅ Clean dataset loaded successfully.")
except FileNotFoundError:
    print("❌ Error: 'senseflow_clean_dataset.csv' not found. Save the file first!")

if 'df' in locals():
    # 2. Total Shape
    print(f"\n[+] Total Verified Rows: {len(df)}")
    print(f"[+] Columns: {list(df.columns)}")

    # 3. Class Distribution (Crucial for Machine Learning)
    print("\n[+] Class Balance (0 = Safe, 1 = Phishing):")
    # This prints the raw counts
    print(df['label'].value_counts())
    # This prints the exact percentages
    print("\n[+] Class Percentages:")
    print(df['label'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

    # 4. Final Text Metrics (The ultimate proof for Phase 2b)
    print("\n[+] Text Length Analytics:")
    print(df['text'].apply(lambda x: len(str(x))).describe())

    print("\n🚀 PHASE 1 & 2 COMPLETE. DATA IS 100% ML-READY.")

=== SENSEFLOW: FINAL DATASET ANALYTICS REPORT ===
✅ Clean dataset loaded successfully.

[+] Total Verified Rows: 16135
[+] Columns: ['clean_text', 'label', 'text']

[+] Class Balance (0 = Safe, 1 = Phishing):
label
1    11512
0     4623
Name: count, dtype: int64

[+] Class Percentages:
label
1    71.35%
0    28.65%
Name: proportion, dtype: object

[+] Text Length Analytics:
count    16135.000000
mean      1274.142919
std       2091.340124
min          1.000000
25%        277.000000
50%        612.000000
75%       1674.500000
max      46332.000000
Name: text, dtype: float64

🚀 PHASE 1 & 2 COMPLETE. DATA IS 100% ML-READY.
